# Module 15: Sensitivity, How Wrong Would the Assumption Have to Be?

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

The estimate rests on an assumption that cannot be verified. The useful
question is not whether it holds, which nobody can answer, but **how badly it
would have to fail before the conclusion changes.**

If the answer is "a violation larger than anything plausible", the result is
robust. If the answer is "a violation the data cannot rule out", say so.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
prof = profile.set_index("agency_id")

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated_ids, outcome="n_uof", offset=None):
    """The standard specification, with whoever is labelled treated."""
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated_ids))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated_ids))
                  & (s["period"] == "phase")).astype(float)
    off = s["lo"] if offset is None else offset
    z = smf.glm(f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson(), offset=off).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z.bse["settled"]

In [ ]:
d["tr"] = d["agency_id"].isin(keep).astype(float)
d["yrc"] = d["yr"] - d["yr"].min()
e0, lo0, hi0, _ = fit(d, keep)
print(f"  the estimate as reported: {e0:+.1f}%  [{lo0:+.1f}, {hi0:+.1f}]")

## 2. A hidden trend difference

Suppose the treated agencies were on a slightly steeper downward path for
reasons nothing in the data records. Subtract a trend of a given size from
them and refit, and see how large it has to be.

In [ ]:
rows = []
for delta in [0.0, -1.0, -2.0, -3.0, -4.0, -5.0]:
    s = d.copy()
    s["adj"] = np.log(s["n_arrests"]) + np.log(1 + delta / 100) * s["tr"] * s["yrc"]
    e, lo, hi, _ = fit(s, keep, offset=s["adj"])
    rows.append({"hidden trend, percent a year": f"{delta:+.1f}",
                 "estimate": f"{e:+.1f}%",
                 "95 percent interval": f"[{lo:+.1f}, {hi:+.1f}]",
                 "still excludes zero": "yes" if not (lo < 0 < hi) else "no"})
pd.DataFrame(rows).set_index("hidden trend, percent a year")

It takes a hidden trend of about **3.4 percent a year** to drive the estimate
to zero, and about **2 percent a year** for the interval to start covering
zero.

## 3. Is that a plausible violation

This is the step people skip. A sensitivity analysis produces a number; it is
only informative next to something that says whether that number is large.

The pre period says what the trend difference actually looks like.

In [ ]:
pre = d[d["period"] == "before"]
z = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", pre,
            family=sm.families.Poisson(), offset=pre["lo"]).fit()
k = [x for x in z.params.index if "yr" in x and "tr" in x][0]
lo, hi = z.conf_int().loc[k]
print(f"  observed pre period trend difference: {pct(z.params[k]):+.2f}% a year")
print(f"  95 percent interval:                  "
      f"[{pct(lo):+.2f}, {pct(hi):+.2f}]\n")
print(f"  needed to zero out the estimate:      about -3.4% a year")
print(f"  needed for the interval to cover zero: about -2.0% a year")

**This is not a reassuring result, and reporting it as one would be wrong.**

The point estimate of the trend difference is 0.70 percent a year, far
smaller than the 3.4 needed. But the interval reaches **3.31 percent a
year**, which is almost exactly the violation that would wipe the estimate
out, and comfortably past the 2.0 that would make the interval cover zero.

The honest summary is: **the pre period is consistent with parallel trends,
and it is also consistent with a violation nearly large enough to explain the
entire result.** Fifty four months of pre period on eleven agencies is not
enough to distinguish those two states of the world.

## 4. A second sensitivity: contamination

The same exercise for the other assumption. How much of the effect would have
to have leaked into the comparison group?

In [ ]:
rng = np.random.default_rng(7)
rows = []
for frac in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    s = d.copy()
    mult = np.where((s["tr"] == 0) & (s["period"] == "after"), 0.88 ** frac, 1.0)
    s["y"] = rng.binomial(s["n_uof"].values.astype(int), np.minimum(mult, 1.0))
    e, lo, hi, _ = fit(s, keep, outcome="y")
    rows.append({"share of the effect reaching every comparison agency":
                     f"{100 * frac:.0f}%",
                 "estimate": f"{e:+.1f}%"})
pd.DataFrame(rows).set_index("share of the effect reaching every comparison agency")

It would take essentially **complete** contamination, every comparison agency
receiving the full effect, to reduce the estimate to nothing.

That is a much more comfortable answer than the trend result, and the reason
is worth noting: it is a claim about seven agencies all quietly running the
same program, which is checkable by asking them, and implausible on its face.

**Two sensitivity analyses, two very different verdicts.** Report both.

## 5. How to write a sensitivity result

> *The estimate is robust to contamination of the comparison group: every
> comparison agency would have to have received the full program effect for
> the estimate to fall to zero. It is less robust to an unmeasured difference
> in trends. A trend difference of 3.4 percent a year favouring the treated
> agencies would eliminate the estimate, and 2.0 percent a year would widen
> the interval to include zero. The observed pre program trend difference is
> 0.70 percent a year, but its 95 percent interval extends to 3.31 percent a
> year, so a violation of the size required cannot be excluded on the
> available pre period.*

Three sentences, and a reader now knows exactly where the result is fragile.

## Exercise

The sensitivity in section 2 applies the hidden trend from the start of the
series. Try applying it only from the year before the program.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    start = d["yr"].max() - 3.0          # roughly the year before the program
    rows = []
    for delta in [-2.0, -3.4, -5.0]:
        for label, ramp in [("from the start of the series", d["yrc"]),
                            ("only from the last pre program year",
                             np.maximum(d["yr"] - start, 0))]:
            s = d.copy()
            s["adj"] = np.log(s["n_arrests"]) + np.log(1 + delta / 100) * s["tr"] * ramp
            e, lo, hi, _ = fit(s, keep, offset=s["adj"])
            rows.append({"hidden trend": f"{delta:+.1f}% a year",
                         "applied": label, "estimate": f"{e:+.1f}%"})
    display(pd.DataFrame(rows).set_index(["hidden trend", "applied"]))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

A trend applied only over the last few years does much less damage than the
same trend applied across the whole series, because the bias accumulates with
the length of time it runs before the intervention.

**That matters for how the sensitivity result is stated.** "A trend
difference of 3.4 percent a year would eliminate the estimate" is only true
for a trend running the full seven years. A violation that began recently
would have to be far larger.

It also suggests a way to narrow the concern: a longer pre period both
improves the parallel trends test and shortens the window in which an
undetected trend could have started without being seen. More pre period data
is worth more than more post period data for this purpose, which is the
opposite of the usual instinct.

</details>

---

**Next:** [Module 16: Writing Up a Causal Claim](Module_16_Writing_Up_A_Causal_Claim.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*